# XLK Clean-Data Logistic Regression Baseline

This notebook establishes a transparent clean-data baseline for the existing XLK next-five-trading-day high-volatility target. It compares a prior-probability dummy classifier with a pre-specified Logistic Regression pipeline. No feature noise, resampling, feature selection or hyperparameter tuning is introduced.

In [ ]:
from pathlib import Path
import os
import warnings

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the cloned repository.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.venv' / '.matplotlib'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'xlk_feature_dataset.csv'
DICTIONARY_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_feature_dictionary.csv'
METRICS_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_logistic_baseline_metrics.csv'
CV_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_logistic_cv_metrics.csv'
COEFFICIENT_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_logistic_coefficients.csv'
CONFUSION_PATH = PROJECT_ROOT / 'outputs' / 'figures' / 'xlk_logistic_confusion_matrix.png'
ROC_PATH = PROJECT_ROOT / 'outputs' / 'figures' / 'xlk_logistic_roc_curve.png'
PR_PATH = PROJECT_ROOT / 'outputs' / 'figures' / 'xlk_logistic_precision_recall_curve.png'

for path in [DATA_PATH, DICTIONARY_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f'Required project input is missing: {path}')
sns.set_theme(style='whitegrid')
print(f'Confirmed project root: {PROJECT_ROOT}')

## 1. Input and assignment validation

The existing target and `train`/`test` labels are read without reconstruction. Predictor names and ordering come from the feature dictionary. The five purged dates must remain absent, and the fixed test period must continue to begin on 10 March 2021.

In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=['Date'])
feature_dictionary = pd.read_csv(DICTIONARY_PATH)
predictors = feature_dictionary['feature_name'].tolist()
target = 'high_volatility'

assert len(predictors) == 11 and len(set(predictors)) == 11, 'Exactly 11 documented predictors are required.'
assert set(predictors).issubset(data.columns), 'Every documented predictor must be present in the dataset.'
assert data['Date'].is_monotonic_increasing and data['Date'].is_unique, 'Dates must be increasing and unique.'
assert data['sample_period'].isin(['train', 'test']).all(), 'Only preserved train and test labels are permitted.'
assert not data[predictors + [target, 'sample_period']].isna().any().any(), 'Inputs must be complete.'
assert np.isfinite(data[predictors].to_numpy(dtype=float)).all(), 'Predictors must be finite.'

original_assignments = data[['Date', 'sample_period']].copy(deep=True)
train_mask = data['sample_period'].eq('train')
test_mask = data['sample_period'].eq('test')
train_data = data.loc[train_mask].copy()
test_data = data.loc[test_mask].copy()
X_train = train_data[predictors]
y_train = train_data[target].astype(int)
X_test = test_data[predictors]
y_test = test_data[target].astype(int)

purged_dates = pd.to_datetime(['2021-03-03', '2021-03-04', '2021-03-05', '2021-03-08', '2021-03-09'])
test_start = pd.Timestamp('2021-03-10')
assert not data['Date'].isin(purged_dates).any(), 'The five purged dates must remain absent.'
assert test_data['Date'].min() == test_start, 'The test period must begin on 10 March 2021.'
assert set(y_train.unique()) == {0, 1} and set(y_test.unique()) == {0, 1}, \
    'Both target classes must exist in the training and test periods.'
pd.testing.assert_frame_equal(data[['Date', 'sample_period']], original_assignments)

print(f'Documented predictors: {len(predictors)}')
print(f'Training observations: {len(train_data):,}')
print(f'Test observations: {len(test_data):,}')
print(f'Test period begins: {test_start.date()}')
print('The existing target, assignments and purge are preserved exactly.')

## 2. Pre-specified models and audited fitting

The primary pipeline standardises predictors and then fits L2-regularised Logistic Regression with `C=1.0`, balanced class weights, the `lbfgs` solver and at most 2,000 iterations. These settings and the 0.5 classification threshold are fixed in advance. Scaling and model estimation use training observations only. The dummy comparator predicts from the training class prior.

In [ ]:
def make_logistic_pipeline():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('logistic', LogisticRegression(
            penalty='l2', C=1.0, class_weight='balanced', solver='lbfgs',
            max_iter=2000, random_state=42,
        )),
    ])

test_dates = set(test_data['Date'])
fit_audit = []

def audited_fit(estimator, X_fit, y_fit, dates, label):
    fit_dates = set(pd.DatetimeIndex(dates))
    overlap = fit_dates.intersection(test_dates)
    assert not overlap, f'Test observations were supplied to fit for {label}: {sorted(overlap)}'
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always', ConvergenceWarning)
        estimator.fit(X_fit, y_fit)
    convergence_messages = [str(item.message) for item in captured if issubclass(item.category, ConvergenceWarning)]
    fit_audit.append({
        'label': label, 'observations': len(X_fit), 'test_overlap': len(overlap),
        'convergence_warnings': len(convergence_messages),
    })
    return estimator, convergence_messages

dummy_model, _ = audited_fit(
    DummyClassifier(strategy='prior'), X_train, y_train, train_data['Date'], 'final_dummy'
)
logistic_model, final_convergence_messages = audited_fit(
    make_logistic_pipeline(), X_train, y_train, train_data['Date'], 'final_logistic'
)
assert not final_convergence_messages, f'Logistic Regression did not converge: {final_convergence_messages}'

scaler = logistic_model.named_steps['scaler']
training_means = X_train.mean().to_numpy(dtype=float)
full_sample_means = data[predictors].mean().to_numpy(dtype=float)
assert np.allclose(scaler.mean_, training_means, rtol=1e-12, atol=1e-12), \
    'Scaler means must equal training predictor means.'
assert not np.allclose(scaler.mean_, full_sample_means, rtol=1e-10, atol=1e-12), \
    'Scaler means must not be fitted from the full sample.'
assert all(item['test_overlap'] == 0 for item in fit_audit), 'Test data must never be supplied to fit.'
print('The final dummy and Logistic Regression models were fitted on training data only.')
print('The Logistic Regression converged without a convergence warning.')
print('Scaler means match training means and differ from full-sample means.')

## 3. Expanding-window temporal-stability diagnostic

An expanding-window `TimeSeriesSplit` with five folds and a five-observation gap is applied within the training period only. Within each fold, the 75th-percentile target threshold is estimated solely from that fold's training `future_rv_5d` observations. The threshold is then applied unchanged to the fold's training and validation observations. These fold-specific thresholds emulate the information available at each historical training endpoint. A fresh complete pipeline is fitted in every fold. The final fixed test evaluation remains based on the single target threshold estimated from the complete pre-test training period. Changing class prevalence across validation periods is evidence of volatility-regime variation rather than something to remove automatically. This exercise diagnoses temporal stability; it is not hyperparameter tuning, and the unchanged fixed test period remains the principal out-of-sample evaluation.

In [ ]:
METRIC_NAMES = [
    'accuracy', 'balanced_accuracy', 'precision', 'recall',
    'f1_score', 'roc_auc', 'average_precision',
]

def calculate_metrics(y_true, predicted_class, probability):
    observed_classes = set(pd.Series(y_true).unique())
    two_class_sample = observed_classes == {0, 1}
    return {
        'accuracy': accuracy_score(y_true, predicted_class),
        'balanced_accuracy': balanced_accuracy_score(y_true, predicted_class) if two_class_sample else np.nan,
        'precision': precision_score(y_true, predicted_class, zero_division=0) if two_class_sample else np.nan,
        'recall': recall_score(y_true, predicted_class, zero_division=0) if two_class_sample else np.nan,
        'f1_score': f1_score(y_true, predicted_class, zero_division=0) if two_class_sample else np.nan,
        'roc_auc': roc_auc_score(y_true, probability) if two_class_sample else np.nan,
        'average_precision': average_precision_score(y_true, probability) if two_class_sample else np.nan,
    }

splitter = TimeSeriesSplit(n_splits=5, gap=5)
cv_rows = []
for fold_number, (fold_train_index, fold_validation_index) in enumerate(splitter.split(X_train), start=1):
    X_fold_train = X_train.iloc[fold_train_index]
    X_fold_validation = X_train.iloc[fold_validation_index]
    fold_training_rv = train_data['future_rv_5d'].iloc[fold_train_index]
    fold_validation_rv = train_data['future_rv_5d'].iloc[fold_validation_index]
    fold_train_dates = train_data['Date'].iloc[fold_train_index]
    fold_validation_dates = train_data['Date'].iloc[fold_validation_index]

    assert fold_train_index.max() + 5 < fold_validation_index.min(), \
        f'CV fold {fold_number} must retain a five-observation gap.'
    assert fold_train_dates.max() < fold_validation_dates.min(), \
        f'Future CV observations must not enter threshold estimation for fold {fold_number}.'
    assert set(fold_train_dates).isdisjoint(test_dates) and set(fold_validation_dates).isdisjoint(test_dates), \
        f'The fixed test period must not enter CV fold {fold_number}.'

    fold_training_threshold = float(fold_training_rv.quantile(0.75))
    independently_recalculated_threshold = float(
        train_data.loc[fold_training_rv.index, 'future_rv_5d'].quantile(0.75)
    )
    assert np.isclose(fold_training_threshold, independently_recalculated_threshold, rtol=0, atol=1e-15), \
        f'Fold {fold_number} threshold must use only that fold training period.'
    y_fold_train = (fold_training_rv > fold_training_threshold).astype(int)
    y_fold_validation = (fold_validation_rv > fold_training_threshold).astype(int)
    assert set(y_fold_train.unique()) == {0, 1}, \
        f'Both classes must exist in the fold {fold_number} training labels.'
    assert y_fold_validation.equals((fold_validation_rv > fold_training_threshold).astype(int)), \
        f'The fold {fold_number} training threshold must be applied unchanged to validation.'

    fold_model, fold_convergence_messages = audited_fit(
        make_logistic_pipeline(), X_fold_train, y_fold_train, fold_train_dates, f'cv_fold_{fold_number}'
    )
    assert not fold_convergence_messages, \
        f'Logistic Regression did not converge in fold {fold_number}: {fold_convergence_messages}'
    fold_probability = fold_model.predict_proba(X_fold_validation)[:, 1]
    fold_prediction = (fold_probability >= 0.5).astype(int)
    assert np.logical_and(fold_probability >= 0, fold_probability <= 1).all(), \
        f'Fold {fold_number} probabilities must lie between zero and one.'
    fold_metrics = calculate_metrics(y_fold_validation, fold_prediction, fold_probability)
    tn, fp, fn, tp = confusion_matrix(y_fold_validation, fold_prediction, labels=[0, 1]).ravel()
    cv_rows.append({
        'result_type': 'fold', 'fold': fold_number,
        'training_start': fold_train_dates.min().date().isoformat(),
        'training_end': fold_train_dates.max().date().isoformat(),
        'validation_start': fold_validation_dates.min().date().isoformat(),
        'validation_end': fold_validation_dates.max().date().isoformat(),
        'training_size': len(fold_train_index), 'validation_size': len(fold_validation_index),
        'fold_training_threshold': fold_training_threshold,
        'training_positive_count': int(y_fold_train.sum()),
        'training_positive_proportion': float(y_fold_train.mean()),
        'validation_positive_count': int(y_fold_validation.sum()),
        'validation_positive_proportion': float(y_fold_validation.mean()),
        'true_negative': int(tn), 'false_positive': int(fp),
        'false_negative': int(fn), 'true_positive': int(tp),
        **fold_metrics,
    })

cv_folds = pd.DataFrame(cv_rows)
cv_mean = cv_folds[METRIC_NAMES].mean()
cv_std = cv_folds[METRIC_NAMES].std(ddof=1)
cv_valid_fold_count = cv_folds[METRIC_NAMES].notna().sum()
summary_rows = []
for result_type, values in [
    ('mean', cv_mean), ('standard_deviation', cv_std), ('valid_fold_count', cv_valid_fold_count)
]:
    row = {column: np.nan for column in cv_folds.columns}
    row.update({'result_type': result_type, 'fold': np.nan, **values.to_dict()})
    summary_rows.append(row)
cv_metrics_table = pd.concat([cv_folds, pd.DataFrame(summary_rows)], ignore_index=True)
available_cv_metrics = cv_folds[METRIC_NAMES].to_numpy(dtype=float)
available_cv_metrics = available_cv_metrics[~np.isnan(available_cv_metrics)]
assert np.isfinite(available_cv_metrics).all(), 'All available CV metrics must be finite.'
assert all(item['test_overlap'] == 0 for item in fit_audit), 'Test data must never be supplied to fit.'
cv_metrics_table.to_csv(CV_PATH, index=False)
print('Fold-specific thresholds and class prevalence:')
print(cv_folds[[
    'fold', 'fold_training_threshold', 'training_positive_count', 'training_positive_proportion',
    'validation_positive_count', 'validation_positive_proportion',
]].to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('Cross-validation fold metrics:')
print(cv_folds[['fold', *METRIC_NAMES]].to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('Cross-validation metric means, standard deviations and valid-fold counts:')
print(pd.DataFrame({
    'mean': cv_mean, 'standard_deviation': cv_std, 'valid_fold_count': cv_valid_fold_count
}).to_string(float_format=lambda value: f'{value:.6f}'))
single_class_folds = cv_folds.loc[
    cv_folds['validation_positive_count'].isin([0])
    | cv_folds['validation_positive_count'].eq(cv_folds['validation_size']), 'fold'
].astype(int).tolist()
if single_class_folds:
    print('Single-class validation folds:', single_class_folds)
    print(
        'Accuracy remains defined and is reported. Balanced accuracy, positive-class precision, '
        'positive-class recall, F1-score, ROC-AUC and average precision are recorded as missing '
        'because these metrics are undefined or not comparable when the validation period contains '
        'no positive observations.'
    )
else:
    print('Both observed target classes occur in every validation fold.')

## 4. Fixed-period test evaluation

Each model is evaluated once on the unchanged test period. Classification uses the pre-specified probability threshold of 0.5; the threshold is not optimised.

In [ ]:
test_rows = []
test_outputs = {}
for model_name, model in [('DummyClassifier', dummy_model), ('LogisticRegression', logistic_model)]:
    probability = model.predict_proba(X_test)[:, 1]
    predicted_class = model.predict(X_test)
    assert np.logical_and(probability >= 0, probability <= 1).all(), \
        f'{model_name} probabilities must lie between zero and one.'
    assert np.array_equal(predicted_class, (probability >= 0.5).astype(int)), \
        f'{model_name} classifications must use the fixed 0.5 threshold.'
    metrics = calculate_metrics(y_test, predicted_class, probability)
    tn, fp, fn, tp = confusion_matrix(y_test, predicted_class, labels=[0, 1]).ravel()
    row = {
        'model': model_name, 'classification_threshold': 0.5, **metrics,
        'true_negative': int(tn), 'false_positive': int(fp),
        'false_negative': int(fn), 'true_positive': int(tp),
    }
    test_rows.append(row)
    test_outputs[model_name] = {
        'probability': probability, 'prediction': predicted_class,
        'confusion_matrix': np.array([[tn, fp], [fn, tp]]),
    }

test_metrics_table = pd.DataFrame(test_rows)
numeric_metrics = test_metrics_table.drop(columns='model').to_numpy(dtype=float)
assert np.isfinite(numeric_metrics).all(), 'All mathematically defined test metrics must be finite.'
expected_logistic_results = {
    'accuracy': 0.7673192771084337,
    'balanced_accuracy': 0.7524657225074762,
    'precision': 0.564755838641189,
    'recall': 0.7189189189189189,
    'f1_score': 0.6325802615933412,
    'roc_auc': 0.8096259098346781,
    'average_precision': 0.6323563262073739,
}
logistic_test_row = test_metrics_table.set_index('model').loc['LogisticRegression']
for metric_name, expected_value in expected_logistic_results.items():
    assert np.isclose(logistic_test_row[metric_name], expected_value, rtol=1e-12, atol=1e-12), \
        f'Fixed-test {metric_name} changed from its preserved value.'
assert (
    int(logistic_test_row['true_negative']), int(logistic_test_row['false_positive']),
    int(logistic_test_row['false_negative']), int(logistic_test_row['true_positive'])
) == (753, 205, 104, 266), 'The fixed-test confusion matrix changed.'
assert all(item['test_overlap'] == 0 for item in fit_audit), 'Test data must never be supplied to fit.'
pd.testing.assert_frame_equal(data[['Date', 'sample_period']], original_assignments)
if METRICS_PATH.exists():
    preserved_test_metrics = pd.read_csv(METRICS_PATH)
    pd.testing.assert_frame_equal(
        test_metrics_table, preserved_test_metrics, check_exact=False, rtol=1e-12, atol=1e-12
    )
else:
    test_metrics_table.to_csv(METRICS_PATH, index=False)
print('Fixed-period test metrics and confusion-matrix counts:')
print(test_metrics_table.to_string(index=False, float_format=lambda value: f'{value:.6f}'))

## 5. Scaled coefficients

Coefficients refer to predictors transformed by the training-fitted `StandardScaler`, so each coefficient represents a one-training-standard-deviation change, holding other predictors fixed. Strong correlations among predictors substantially limit isolated coefficient interpretation; coefficients must not be treated as independent causal effects or standalone importance measures.

In [ ]:
coefficient_table = pd.DataFrame({
    'feature_name': predictors,
    'coefficient': logistic_model.named_steps['logistic'].coef_[0],
}).merge(
    feature_dictionary[['feature_name', 'feature_group']], on='feature_name', how='left', validate='one_to_one'
)
coefficient_table['absolute_coefficient'] = coefficient_table['coefficient'].abs()
coefficient_table = coefficient_table[[
    'feature_name', 'feature_group', 'coefficient', 'absolute_coefficient'
]].sort_values('absolute_coefficient', ascending=False, ignore_index=True)
assert coefficient_table['feature_group'].notna().all(), 'Every coefficient must have a documented feature group.'
assert np.isfinite(coefficient_table[['coefficient', 'absolute_coefficient']].to_numpy()).all(), \
    'All scaled coefficients must be finite.'
if COEFFICIENT_PATH.exists():
    preserved_coefficients = pd.read_csv(COEFFICIENT_PATH)
    pd.testing.assert_frame_equal(
        coefficient_table, preserved_coefficients, check_exact=False, rtol=1e-12, atol=1e-12
    )
else:
    coefficient_table.to_csv(COEFFICIENT_PATH, index=False)
print('Scaled Logistic Regression coefficients, ordered by absolute magnitude:')
print(coefficient_table.to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('Warning: strong predictor correlations limit isolated coefficient interpretation.')

## 6. Fixed-test diagnostic figures

The confusion matrix reports Logistic Regression counts at the fixed 0.5 threshold. ROC and precision–recall curves show both the primary model and the dummy comparator on the same unchanged test period.

In [ ]:
logistic_confusion = test_outputs['LogisticRegression']['confusion_matrix']
fig, ax = plt.subplots(figsize=(7, 6))
display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=logistic_confusion,
    display_labels=['Normal volatility (0)', 'High volatility (1)'],
)
display_matrix.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title('XLK Logistic Regression Test Confusion Matrix')
ax.set_xlabel('Predicted class')
ax.set_ylabel('Actual class')
fig.tight_layout()
if not CONFUSION_PATH.exists():
    fig.savefig(CONFUSION_PATH, dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
for model_name, label in [('LogisticRegression', 'Logistic Regression'), ('DummyClassifier', 'Dummy classifier')]:
    probability = test_outputs[model_name]['probability']
    false_positive_rate, true_positive_rate, _ = roc_curve(y_test, probability)
    auc_value = roc_auc_score(y_test, probability)
    ax.plot(false_positive_rate, true_positive_rate, linewidth=2, label=f'{label} (ROC-AUC = {auc_value:.3f})')
ax.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Chance reference')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('XLK Fixed-Test ROC Curves')
ax.set_xlabel('False-positive rate')
ax.set_ylabel('True-positive rate')
ax.legend(loc='lower right')
fig.tight_layout()
if not ROC_PATH.exists():
    fig.savefig(ROC_PATH, dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
for model_name, label in [('LogisticRegression', 'Logistic Regression'), ('DummyClassifier', 'Dummy classifier')]:
    probability = test_outputs[model_name]['probability']
    precision_values, recall_values, _ = precision_recall_curve(y_test, probability)
    ap_value = average_precision_score(y_test, probability)
    ax.plot(recall_values, precision_values, linewidth=2, label=f'{label} (average precision = {ap_value:.3f})')
ax.axhline(y_test.mean(), linestyle='--', color='grey', label='Test prevalence')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('XLK Fixed-Test Precision–Recall Curves')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(loc='best')
fig.tight_layout()
if not PR_PATH.exists():
    fig.savefig(PR_PATH, dpi=300, bbox_inches='tight')
plt.show()

print('Validated the fixed-test figures without recreating existing files.')
print('All baseline quality checks passed; the test set was evaluated once and was never used for fitting.')

## Interpretation and limitations

This is a pre-specified clean-data baseline rather than a tuned forecasting system. The expanding-window results assess temporal variability within the training period, while the fixed test period provides the principal out-of-sample result. Class weighting changes the fitting objective, so the resulting probabilities should not automatically be interpreted as calibrated event probabilities. Overlapping adjacent target windows and correlated predictors also limit naive inferential interpretation and should be addressed in the final validation design.